Task 1: Build the data foundation
Before connecting Python to BigQuery, you need production data. Run this DDL script in the BigQuery console to create the UrbanMart dataset with two years of transaction history across five product categories.

NOTE
Copy this entire block and paste it into the BigQuery SQL editor. Execute it as a single script. BigQuery processes each statement sequentially. The first command creates your dataset namespace. The second generates a product catalog with 100 items distributed across five categories. The third simulates 10,000 customer orders spanning 730 days back from today.

In [ ]:
# sql_engine: bigquery
# output_variable: df
# start _sql
_sql = """

""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
df = _bqsqlcell.run(_sql)
df

In [ ]:
# sql_engine: bigquery
# output_variable: df
# start _sql
_sql = """
-- Create the dataset
CREATE SCHEMA IF NOT EXISTS `urbanmart_raw`;

-- Generate Product Catalog
CREATE OR REPLACE TABLE `urbanmart_raw.products` AS
SELECT
  product_id,
  CASE MOD(product_id, 5)
    WHEN 0 THEN 'Electronics'
    WHEN 1 THEN 'Home & Garden'
    WHEN 2 THEN 'Fashion'
    WHEN 3 THEN 'Sports'
    ELSE 'Toys'
  END AS category,
  ROUND(10 + RAND() * 90, 2) AS price
FROM UNNEST(GENERATE_ARRAY(1, 100)) AS product_id;

-- Generate Transaction History
CREATE OR REPLACE TABLE `urbanmart_raw.transactions` AS
SELECT
  GENERATE_UUID() AS order_id,
  DATE_ADD(CURRENT_DATE(), INTERVAL -CAST(FLOOR(RAND() * 730) AS INT64) DAY) AS order_date,
  CAST(FLOOR(1 + RAND() * 100) AS INT64) AS product_id,
  CAST(FLOOR(1 + RAND() * 5) AS INT64) AS quantity
FROM UNNEST(GENERATE_ARRAY(1, 10000));
SELECT COUNT(*) FROM urbanmart_raw.transactions;
/* in BigQuery. You should see 10,000 rows.*/
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
df = _bqsqlcell.run(_sql)
df

Task 2: Connect Python to BigQuery
This video demonstrates how to easy connect Google BigQuery to a Python notebook using the BigQuery API and Google Colab. Google Colab allows an easy integration with the Google BigQuery API allowing SQL queries with one line of code.

In [ ]:
from google.cloud import bigquery

client = bigquery.Client()


Task 3: Extract category performance data
The VP wants to see which categories generate the most revenue. Write Python code that queries BigQuery and loads results directly into a Pandas DataFrame.

HINT
Write your SQL query as a multi-line string in Python using triple quotes. The query should join the transactions table with the products table to access both quantity and price data. Calculate total revenue as quantity multiplied by price, then group by category.

Use the client's query() method to execute the SQL, then chain .to_dataframe() to convert results into a Pandas DataFrame df_category . This method uses BigQuery's Storage API for efficient data transfer.

In [ ]:

  # Define the SQL query
query = """
SELECT
  p.category,
  SUM(t.quantity * p.price) AS total_revenue
FROM `urbanmart_raw.transactions` t
JOIN `urbanmart_raw.products` p
  ON t.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC
"""


# Execute query and load into DataFrame
df_category = client.query(query).to_dataframe()

# Display results
print(df_category)


        category  total_revenue
0  Home & Garden      348240.91
1        Fashion      338840.18
2    Electronics      331973.71
3         Sports      317503.05
4           Toys      268691.06


"""
Task 4: Use parameterized queries for security
The VP wants to filter results dynamically based on user input. She might ask for Electronics revenue one day and Sports the next. You cannot use f-strings to insert variables into SQL because this creates SQL injection vulnerabilities.

Here use target_category = "Electronics" and min_revenue_threshold = 5000 to demonstrate parameterized queries.

HINT
BigQuery uses @parameter_name syntax for parameterized queries. Define your SQL with @ placeholders where variables should go. Create a QueryJobConfig object and populate its query_parameters list with ScalarQueryParameter objects. Each parameter needs a name, data type, and value.
"""

In [ ]:


#déclaration
target_category = "Electronics"
min_revenue_threshold = 500

#la requête que je stocke dans une variable
parameterized_query = """
SELECT
  order_date,
  SUM(t.quantity * p.price) AS daily_revenue
FROM `urbanmart_raw.transactions` t
JOIN `urbanmart_raw.products` p
  ON t.product_id = p.product_id
WHERE p.category = @category
 /* SQL injection => éviter de récupérer directement l'input du user
 en séparant le code SQL des données => BQ s'en chargera*/
GROUP BY order_date
HAVING daily_revenue >= @min_revenue
ORDER BY order_date
"""
# configuration et mapping des paramètres
job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter(
            #1er mapping
            "category", "STRING", target_category #
        ),
        bigquery.ScalarQueryParameter(
            "min_revenue", "FLOAT64", min_revenue_threshold
        ),
    ]
)
#requête finale
df_filtered = client.query(
    parameterized_query,
    job_config=job_config
).to_dataframe()

print(
    f"Found {len(df_filtered)} days where "
    f"{target_category} exceeded ${min_revenue_threshold}"
)
print(df_filtered.head())




Found 275 days where Electronics exceeded $500
   order_date  daily_revenue
0  2023-12-21         517.48
1  2023-12-23         895.62
2  2023-12-27         750.74
3  2023-12-28         609.49
4  2023-12-29         847.16


Task 5: Calculate moving averages in Python
Raw daily revenue fluctuates wildly. Weekend dips and holiday spikes obscure the underlying trend. The VP needs smooth trend lines to make inventory decisions. You calculate a 30-day moving average for each category.

HINT
First, query all daily revenue data grouped by category and date. Use a comprehensive date range to ensure you have enough history for the moving average calculation.

Once you have the DataFrame df_daily, use Pandas groupby() to process each category separately. Apply the rolling() method with a window of 30 days to calculate the moving average. The min_periods=1 parameter ensures you get values even for the first 29 days.
Then reset the index reset_index() to align the moving average back to the original DataFrame structure.

The moving_avg_30d column now shows smoothed trends. If Electronics shows a declining 30-day average for three consecutive weeks, that signals real trouble rather than random noise.

In [ ]:
#extraction du rev moyen journalier
full_query = """
SELECT
  p.category,
  t.order_date,
  SUM(t.quantity * p.price) AS total_revenue
FROM `urbanmart_raw.transactions` t
JOIN `urbanmart_raw.products` p
  ON t.product_id = p.product_id
GROUP BY p.category, t.order_date
ORDER BY p.category, t.order_date
"""


In [ ]:
# SQL => dataframe
df_daily = client.query(full_query).to_dataframe()


In [ ]:
# moyenne mobile sur 30 jours avec .sort_values().groupBy().rolling().mean()


df_daily['moving_avg_30d'] = (
    df_daily
    .sort_values('order_date')
    .groupby('category')['total_revenue']
    .rolling(window=30, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)
print(df_daily.tail(20))


     category  order_date  total_revenue  moving_avg_30d
3399     Toys  2025-11-28         254.82      339.994000
3400     Toys  2025-11-29         575.39      351.770333
3401     Toys  2025-11-30         101.10      352.295667
3402     Toys  2025-12-01         435.24      345.752333
3403     Toys  2025-12-02         307.14      342.265667
3404     Toys  2025-12-03         278.60      344.590667
3405     Toys  2025-12-04          48.74      335.448667
3406     Toys  2025-12-05         182.49      338.747000
3407     Toys  2025-12-08         248.90      338.306333
3408     Toys  2025-12-09         207.16      322.810000
3409     Toys  2025-12-10         154.95      289.831333
3410     Toys  2025-12-11         452.01      287.393000
3411     Toys  2025-12-12          71.93      276.502667
3412     Toys  2025-12-13         449.35      289.566000
3413     Toys  2025-12-14         314.35      286.829000
3414     Toys  2025-12-15         934.78      299.560667
3415     Toys  2025-12-16      

Task 6: Write results back to BigQuery
Your analysis reveals that Sports revenue is accelerating while Fashion is declining. The VP needs this insight in BigQuery so other teams can access it through their own dashboards and reports. You push the DataFrame back to a new table.

HINT
Use load_table_from_dataframe() to write data back to BigQuery. Specify the destination table using the format project.dataset.table_name. Create a LoadJobConfig object to control write behavior. WRITE_TRUNCATE replaces the table completely, while WRITE_APPEND adds rows to existing data.

Call .result() on the job object to wait for completion. This ensures the upload finishes before your script continues.


Check if the data uploaded successfully.

Verify the upload by running SELECT * FROM urbanmart_raw.category_trends LIMIT 10; in the BigQuery console. Your Python-calculated moving averages now live in the data warehouse where SQL-based tools can access them.

In [ ]:
#création d'une nouvelle table
destination_table = "urbanmart_raw.category_trends"


In [ ]:
# accès en écriture
load_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)
load_job = client.load_table_from_dataframe(
    df_daily,
    destination_table,
    job_config=load_config
)

load_job.result()

print(
    f"Loaded {load_job.output_rows} rows "
    f"to {destination_table}"
)


Loaded 3419 rows to urbanmart_raw.category_trends


In [ ]:
# sql_engine: bigquery
# output_variable: df
# start _sql
_sql = """
WITH filtered_categories AS (
    SELECT
        category,
        order_date,
        moving_avg_30d
    FROM urbanmart_raw.category_trends
    WHERE category IN ('Sports', 'Fashion')
)
SELECT
    category,
    order_date,
    moving_avg_30d,
    moving_avg_30d
      - LAG(moving_avg_30d) OVER (
            PARTITION BY category
            ORDER BY order_date
        ) AS moving_avg_delta
FROM filtered_categories
ORDER BY order_date DESC;

""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
df = _bqsqlcell.run(_sql)
df

TableWidget(page_size=10, row_count=1371, table_html='<table border="1" class="dataframe table table-striped t…

Computation deferred. Computation will process 66.5 kB

Task 7: Build the visualizations
The VP wants a presentation-ready. Use Plotly to create interactive line charts showing both daily revenue and 30-day moving averages for each category.

HINT
Use Plotly Express px.line() to create line charts. Set the x-axis to order_date, y-axis to total_revenue, and color by category to differentiate lines. Add the moving average as a separate line by calling add_scatter() on the figure object.

In [ ]:
import plotly.express as px

fig = px.line(
    df_daily,
    x='order_date',
    y='total_revenue',
    color='category',
    title='Revenu journalier par catégorie'
)
fig.add_scatter(
    x=df_daily['order_date'],
    y=df_daily['moving_avg_30d'],
    mode='lines',
    name='Moyenne mobile 30 jours',
    line=dict(dash='dash')
)
fig.show()
